<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/02-tool-design.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 工具设计

**目标：** 设计出模型真正能用好用的工具。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端，这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 工具质量胜过 prompt 调优

当 agent 行为不端时，本能反应是去改 system prompt。通常真正的问题出在工具上。有两个框架要记住：

- **工具描述就是一个 prompt。** 它在每一次请求里都会被注入模型上下文，而且是模型关于你这个工具唯一会读到的文档。
- **工具接口是一个调用方为模型的 API。** 你所知道的关于 API 设计的一切依然适用，只是你的消费方在每次调用时都要重读文档、没法单步跟踪你的代码，还会逐字理解错误消息。

本 notebook 用四对「改前/改后」来演示这一点：用 notebook 01 里同一个循环、同一个任务，分别配一个*坏*工具和一个*好*工具。两次运行之间唯一变化的就是工具。

## 循环（notebook 01 的紧凑版复制）

同一个循环，用工具列表和一个实现 dict 参数化，现在还会返回总的 token 用量。我们需要这个来计算第 3 对里坏设计的代价。

In [ ]:
import json

def run_agent(task, tools, impls, max_turns=8, verbose=True):
    """notebook 01 的循环，紧凑版。返回 (final_text, usage_totals)。"""
    messages = [{"role": "user", "content": task}]
    usage = {"input_tokens": 0, "output_tokens": 0}

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages,
        )
        usage["input_tokens"] += response.usage.prompt_tokens
        usage["output_tokens"] += response.usage.completion_tokens
        choice = response.choices[0]
        msg = choice.message

        if choice.finish_reason != "tool_calls":
            text = msg.content or ""
            if verbose:
                print(f"[done in {turn} turn(s)] {text.strip()[:300]}")
                print(f"[usage] {usage}")
            return text, usage

        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = impls.get(tc.function.name)
            try:
                out = str(fn(**args)) if fn else f"Error: unknown tool '{tc.function.name}'."
            except TypeError as e:
                out = f"Error: bad arguments: {e}"
            if verbose:
                print(f"[turn {turn}] {tc.function.name}({tc.function.arguments[:90]}) -> {out.strip()[:90]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    if verbose:
        print(f"[stopped: hit max_turns={max_turns}]")
    return "(hit max_turns)", usage

# 每个示例共用的假数据：一个极小的订单系统。
ORDERS = {
    "ORD-4521": {"status": "shipped", "carrier": "UPS", "eta": "2026-08-15",
                 "customer_id": "C-102", "placed": "2026-08-08"},
    "ORD-4488": {"status": "delivered", "carrier": "FedEx", "eta": "2026-07-30",
                 "customer_id": "C-102", "placed": "2026-07-25"},
    "ORD-4530": {"status": "processing", "carrier": None, "eta": None,
                 "customer_id": "C-217", "placed": "2026-08-11"},
}
CUSTOMERS = {
    "C-102": {"name": "Dana Wong", "email": "dana@example.com", "orders": ["ORD-4488", "ORD-4521"]},
    "C-217": {"name": "Sam Patel", "email": "sam@example.com", "orders": ["ORD-4530"]},
}


## 第 1 对：描述就是一个 prompt

下面两个版本包装的是*同一个*查询函数。坏的那个用四个字描述工具做什么。好的那个说明了**什么时候用它、什么时候别用、精确的输入格式，以及会返回什么**：这是一次就能让模型正确调用的四个要点。

In [ ]:
def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return f"No order found for id '{order_id}'."
    return (f"status={order['status']} carrier={order['carrier']} "
            f"eta={order['eta']} placed={order['placed']}")

BAD_DESC_TOOL = [{
    "type": "function",
    "function": {
        "name": "query",
        "description": "Query the system.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
}]

GOOD_DESC_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order_status",
        "description": (
            "Look up the shipping status of a single order. Use this when the user asks "
            "where an order is or when it will arrive. Do NOT use it for refunds, "
            "inventory, or customer profiles. order_id must be the full ID in the form "
            "'ORD-' plus digits, e.g. 'ORD-4521'. Returns status, carrier, ETA, and the "
            "date the order was placed."
        ),
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string",
                                        "description": "Full order ID, e.g. 'ORD-4521'."}},
            "required": ["order_id"],
        },
    },
}]

TASK_1 = "Where is order 4521 and when will it arrive?"

print("--- bad description ---")
run_agent(TASK_1, BAD_DESC_TOOL, {"query": lookup_order})

print("\n--- good description ---")
run_agent(TASK_1, GOOD_DESC_TOOL, {"get_order_status": lookup_order})


跑一下，比较两次 trace。用含糊的工具时，模型得*猜* ID 格式。它常常传入 `4521`，得到「no order found」，然后白白烧掉一到两轮去试探各种变体；有时它半猜半蒙，带着模棱两可的不确定性作答。用好的描述时，描述里的格式示例意味着第一次调用通常就是 `ORD-4521`，运行两轮就结束。

实现没有任何改动。光是描述就换来了正确*和*更少的付费轮次。

## 第 2 对：一个工具只干一件事

下面这个巨型工具是一个真实的反模式：一个入口点，一个 `action` 字符串，外加八个可选参数，而它们的有效性取决于 action。每一次调用都是一次选错组合的机会。替代方案是三个聚焦的工具，schema 本身就使非法调用不可能发生。

In [ ]:
def customer_data(action=None, customer_id=None, order_id=None, field=None,
                  value=None, query=None, limit=None, sort=None, format=None):
    # 一个函数，多种失败模式 —— 正是模型所经历的东西。
    if action == "get_customer":
        if not customer_id:
            return "Error: action 'get_customer' requires customer_id."
        c = CUSTOMERS.get(customer_id)
        return str(c) if c else f"Error: no customer '{customer_id}'."
    if action == "list_orders":
        if not customer_id:
            return "Error: action 'list_orders' requires customer_id."
        c = CUSTOMERS.get(customer_id)
        if not c:
            return f"Error: no customer '{customer_id}'."
        return ", ".join(c["orders"])
    if action == "get_order":
        if not order_id:
            return "Error: action 'get_order' requires order_id."
        o = ORDERS.get(order_id)
        return str(o) if o else f"Error: no order '{order_id}'."
    return f"Error: unknown action '{action}'. Valid: get_customer, list_orders, get_order."

MEGA_TOOL = [{
    "type": "function",
    "function": {
        "name": "customer_data",
        "description": "Access customer and order data. Set action and the relevant parameters.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string"},
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"},
                "field": {"type": "string"},
                "value": {"type": "string"},
                "query": {"type": "string"},
                "limit": {"type": "integer"},
                "sort": {"type": "string"},
            },
            "required": [],
        },
    },
}]

def get_customer(customer_id):
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"Error: no customer '{customer_id}'."
    return f"name={c['name']} email={c['email']}"

def list_orders(customer_id):
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"Error: no customer '{customer_id}'."
    lines = [f"{oid}: placed {ORDERS[oid]['placed']}" for oid in c["orders"]]
    return "\n".join(lines)

def get_order(order_id):
    o = ORDERS.get(order_id)
    if not o:
        return f"Error: no order '{order_id}'."
    return f"status={o['status']} carrier={o['carrier']} eta={o['eta']}"

def _focused(name, description, props, required):
    return {"type": "function",
            "function": {"name": name, "description": description,
                         "parameters": {"type": "object", "properties": props,
                                        "required": required}}}

FOCUSED_TOOLS = [
    _focused("get_customer",
             "Get a customer's name and email by customer_id (e.g. 'C-102').",
             {"customer_id": {"type": "string"}}, ["customer_id"]),
    _focused("list_orders",
             "List a customer's order IDs with the date each was placed, "
             "newest and oldest included. Use get_order for details on one order.",
             {"customer_id": {"type": "string"}}, ["customer_id"]),
    _focused("get_order",
             "Get status, carrier, and ETA for one order by full order ID (e.g. 'ORD-4521').",
             {"order_id": {"type": "string"}}, ["order_id"]),
]

TASK_2 = "Has customer C-102's most recent order shipped yet?"

print("--- mega-tool ---")
run_agent(TASK_2, MEGA_TOOL, {"customer_data": customer_data})

print("\n--- three focused tools ---")
run_agent(TASK_2, FOCUSED_TOOLS,
          {"get_customer": get_customer, "list_orders": list_orders, "get_order": get_order})


跑一下，注意每次 trace 的形状。巨型工具那次通常*最终*也能成（模型很擅长恢复），但要看有没有浪费的轮次：某个缺参数的调用、某个凭空编造的 `action` 名、某次在错误字符串之后的重试。聚焦版本往往是一条干净的 `list_orders` → `get_order` 链，因为每个 schema 只允许合理的调用。

经验法则：如果你在一个工具里写 `if action == ...` 分发，那些 action 就该是各自独立的工具。enum 和必填字段是模型迈不过去的护栏；而一个 docstring 约定不是。

## 第 3 对：只返回下一个决策所需的东西

这一对既关乎钱，也关乎准确率。工具结果不是只被读一次的。它会落进对话里，并在循环中**随之后的每一次请求重发**。一个 5,000 token 的 JSON 大转储在第 2 轮返回、总共 6 轮运行，就会作为输入再被计费大约四次。

下面，原始工具返回整条订单记录：40 个行项目、一份审计日志、内部标志。提炼过的工具只返回任务实际需要的四个字段。

In [ ]:
# 构造一条刻意臃肿的订单记录。
BIG_ORDER = {
    "order_id": "ORD-4521",
    "status": "shipped",
    "carrier": "UPS",
    "eta": "2026-08-15",
    "internal_flags": {"fraud_score": 0.02, "warehouse": "EWR-4", "priority_class": "B2",
                       "reprice_pending": False, "legacy_migration_id": "LM-889231"},
    "line_items": [
        {"sku": f"SKU-{i:04d}", "description": f"Replacement part, model X-{i}, rev {i % 7}",
         "qty": (i % 3) + 1, "unit_price_cents": 950 + 137 * i,
         "tax_code": "TX-STD", "fulfillment_node": "EWR-4"}
        for i in range(40)
    ],
    "audit_log": [
        {"ts": f"2026-08-08T{h:02d}:14:03Z", "actor": "system",
         "event": "state_transition", "detail": f"step_{h} completed, queue depth {h * 3}"}
        for h in range(20)
    ],
}
BIG_ORDER["total_cents"] = sum(li["qty"] * li["unit_price_cents"] for li in BIG_ORDER["line_items"])

def get_order_raw(order_id):
    if order_id != "ORD-4521":
        return f"Error: no order '{order_id}'."
    return json.dumps(BIG_ORDER)  # 每次都是完整的转储

def get_order_summary(order_id):
    if order_id != "ORD-4521":
        return f"Error: no order '{order_id}'."
    o = BIG_ORDER
    return (f"order_id={o['order_id']} status={o['status']} eta={o['eta']} "
            f"item_count={len(o['line_items'])} total=${o['total_cents'] / 100:.2f}")

RAW_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Get an order record by full order ID (e.g. 'ORD-4521').",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]},
    },
}]
SUMMARY_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Get an order's status, ETA, item count, and total value by full "
                       "order ID (e.g. 'ORD-4521').",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]},
    },
}]

TASK_3 = "What's the total value of order ORD-4521, and has it shipped?"

print("--- raw dump ---")
_, usage_raw = run_agent(TASK_3, RAW_TOOL, {"get_order": get_order_raw})

print("\n--- distilled ---")
_, usage_distilled = run_agent(TASK_3, SUMMARY_TOOL, {"get_order": get_order_summary})

print("\n--- token bill ---")
print(f"raw:       {usage_raw}")
print(f"distilled: {usage_distilled}")


跑一下，比较两行 `usage`。两个版本都能得到正确答案；总额在记录里已经预先算好了，所以原始版本甚至不需要做算术。但原始那次的输入 token 通常是提炼版的数倍，而这只是*一次*冗长的结果配上一个*很短*的任务。现在把它放大：一个 20 轮的 agent，它的搜索工具返回原始 API 载荷，那么在剩下的每一轮里都要为这些载荷付费。冗长的返回会复利增长。

还有准确率的代价，不只是 token 代价：「has it shipped?」的答案只是埋在 40 行噪声里的一个字段。提炼是在帮模型的注意力一把。

设计上的做法：确定*下一个决策*需要什么，就精确返回那些。如果某些调用方真的需要行项目，那是一个第二个工具（`get_order_items`）或一个 `fields` enum，而不是默认开闸放水。

## 第 4 对：能教会模型的错误

工具会失败。问题是模型能拿这个失败做什么。`"Error"` 是死路；一个说明**哪里错了、该试什么**的错误就是一次纠正航向。这里两个工具有*同样极简的描述*（只有错误字符串不同），而任务用了一个格式错误的 ID，所以第一次调用注定失败。

In [ ]:
def get_order_bad_errors(order_id):
    o = ORDERS.get(order_id)
    if not o:
        return "Error"
    return f"status={o['status']} carrier={o['carrier']} eta={o['eta']}"

def get_order_good_errors(order_id):
    o = ORDERS.get(order_id)
    if not o:
        digits = "".join(ch for ch in order_id if ch.isdigit())
        hint = f" Did you mean 'ORD-{digits}'?" if digits and f"ORD-{digits}" in ORDERS else ""
        return (f"Error: no order with id '{order_id}'. Order IDs are 'ORD-' followed "
                f"by digits, e.g. 'ORD-4521'.{hint} Call get_order again with the full ID.")
    return f"status={o['status']} carrier={o['carrier']} eta={o['eta']}"

# 两者拥有完全相同、刻意极简的描述 —— 错误消息是这个
# 实验里唯一的变量。
MINIMAL_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Look up an order by ID.",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]},
    },
}]

TASK_4 = "Where is order 4521?"

print("--- error says 'Error' ---")
run_agent(TASK_4, MINIMAL_TOOL, {"get_order": get_order_bad_errors})

print("\n--- error teaches ---")
run_agent(TASK_4, MINIMAL_TOOL, {"get_order": get_order_good_errors})


跑一下，看每次 trace 的第二轮。面对光秃秃的 `"Error"`，模型是瞎的：它可能重复同一个调用、猜各种格式，或者直接放弃并告诉用户找不到这个订单。有了能教它的错误，紧接着的那次调用就是 `ORD-4521`，任务完成，因为工具告诉了模型如何自我纠正。

好的错误消息写起来很便宜，却在每一次失败上永远地回报。包含三件事：收到了什么、为什么它不合法，以及一个具体的下一步（最好带个例子）。

## 规则，提炼版

| 规则 | 坏 | 好 |
|---|---|---|
| **名字说明它做什么** | `query`、`process`、`handle_data` | `get_order_status`、`list_orders` |
| **描述 = 何时用、何时别用、输入格式、返回什么** | 「查询系统。」 | 「当用户问某个订单在哪里时使用。不用于退款。order_id 形如 'ORD-4521'。返回 status、carrier、ETA。」 |
| **一个工具只干一件事** | 一个工具，`action` 参数，8 个可选字段 | 三个工具，各自 1–2 个必填参数 |
| **有类型、有约束的参数** | `options: string`，靠约定解析 | `required` 字段、固定取值用 `enum`、逐属性描述 |
| **只返回下一个决策所需的东西** | 完整 JSON 转储，审计日志全都塞进来 | 任务用到的那 3–5 个字段；细节用单独的工具 |
| **错误能教学** | `"Error"` | 收到了什么、为何失败、该试什么 |

### 到处都是一样的对称性

如果你要搭 MCP server 或把内部 API 暴露给 agent，这就是这份工作的全部：你在为一个消费方做 API 设计，而这个消费方在**每一次调用**时都会重读文档，从不跨会话积累经验。这推翻了一些习惯（文档不是读一次就够的附属品，它是热路径），但也意味着你对描述或错误消息的每一处改进都会立刻交付给每个调用方。没有客户端代码需要更新。

## 练习

1. **加一个 enum 护栏。** 给 `list_orders` 加一个 `sort` 参数，形如 `{"type": "string", "enum": ["newest_first", "oldest_first"]}`，并实现它。然后用把约定埋在描述里的自由字符串做同一个功能：各跑五次「show me Dana's orders, oldest first」，数一数格式错误。
2. **不拆分就修好巨型工具。** 让 `customer_data` 保持为一个工具，但把 `action` 改成 enum，在描述里记录每个 action 需要哪些参数，并让每条错误消息都点名缺失的参数。重跑 TASK_2。一个规定良好的巨型工具离聚焦版本有多近？
3. **做一个 `fields` 选择器。** 改 `get_order`，让它接受一个可选的 `fields: enum 数组` 并只返回那些字段。用用量计数器验证：在臃肿订单上只要 `["status"]` 的代价只是完整转储的一小部分。
4. **写出「教学式错误」工具箱。** 拿 notebook 01 的文件系统工具，升级每一条失败路径：对不存在路径的 `read_file` 应当提示最接近的现存路径（试试 `difflib.get_close_matches`），`calculator` 遇到坏表达式应当指出哪个字符违反了规则。在任务里埋一个错别字，重跑 notebook 01 的任务。
